# ☁️ Notebook 06: AWS Cloud Foundations & The $1.00 Budget Shield
**Cloud & Software Developer Mastery · Day 01**

Welcome to the Cloud portion of Day 01!

Now that you have solid Python superpowers, let's explore **what the Cloud actually is**, why companies build on AWS, and how you can experiment with complete confidence without ever having to worry about spending money.

In this notebook, we will:
1. Understand the Cloud through the friendly **Electric Power Grid** analogy.
2. Explore AWS Global & India Infrastructure (Mumbai `ap-south-1` & Hyderabad `ap-south-2`).
3. Learn who manages what using the **Landlord & Tenant** Shared Responsibility Model.
4. Deploy the **Zero-Spend $1.00 Budget Shield** using Python.
5. Generate an official AWS Budgets JSON configuration!


### 💡 Step 1: What is the Cloud? (The Power Grid Mental Model)

In the 1800s, when a factory opened, it had to build its own private steam engine, coal shed, and generator. It required a team of full-time mechanics just to generate electricity.

When Thomas Edison and Nikola Tesla built the centralized AC power grid, factories tore down their coal engines and plugged into the wall socket. They paid only for the kilowatt-hours they used.

**Amazon Web Services (AWS) is the planetary electric grid of computational power:**
- You don't buy physical computers, pour concrete, or buy diesel generators.
- You plug into AWS via code and APIs.
- You run your software and pay strictly for the seconds of compute you consume!


In [ ]:
# Comparing On-Premise Datacenter vs Cloud Utility Model
def compare_hosting(physical_server_cost, power_and_cooling_annual, cloud_hourly_rate):
    hours_per_year = 24 * 365 # 8760 hours
    
    on_prem_cost = physical_server_cost + power_and_cooling_annual
    cloud_cost = cloud_hourly_rate * hours_per_year
    savings = on_prem_cost - cloud_cost
    
    return {
        "On-Premise (CapEx)": f"${on_prem_cost:,.2f} USD",
        "AWS Cloud (OpEx)": f"${cloud_cost:,.2f} USD",
        "First Year Savings": f"${savings:,.2f} USD ({round((savings/on_prem_cost)*100)}% cheaper!)"
    }

print("Hosting Cost Comparison:")
for k, v in compare_hosting(5000.0, 1800.0, 0.0116).items():
    print(f"• {k:<20}: {v}")


---
### 🌐 Step 2: Regions & Availability Zones (India Deep Dive)
- **AWS Region**: A separate geographic area (e.g. `ap-south-1` Mumbai, `ap-south-2` Hyderabad).
- **Availability Zone (AZ)**: Discrete physical data centers inside a region, separated by 10 to 100 kilometers, linked by high-speed dark fiber (< 1.5ms latency).
- In India, having **both Mumbai and Hyderabad** allows banks and healthcare systems to achieve 100% in-country Disaster Recovery without sensitive customer data ever leaving the country!


In [ ]:
# Modeling Indian Cloud Infrastructure in Python
aws_india_infrastructure = {
    "ap-south-1": {
        "city": "Mumbai",
        "launched": 2016,
        "availability_zones": ["ap-south-1a", "ap-south-1b", "ap-south-1c"],
        "role": "Primary Commercial & Financial Capital Anchor"
    },
    "ap-south-2": {
        "city": "Hyderabad",
        "launched": 2022,
        "availability_zones": ["ap-south-2a", "ap-south-2b", "ap-south-2c"],
        "role": "Secondary In-Country Disaster Recovery & Gov Anchor"
    }
}

for region_code, info in aws_india_infrastructure.items():
    print(f"\nRegion: {region_code} ({info['city']})")
    print(f"  Role: {info['role']}")
    print(f"  AZs : {', '.join(info['availability_zones'])}")


---
### 🤝 Step 3: The Shared Responsibility Model (The Landlord & Tenant Analogy)
Think of renting an apartment:
- **The Landlord (AWS)**: Responsible for the building foundations, exterior walls, roof, elevators, and front gate security (**Security OF the Cloud**).
- **The Tenant (You)**: Responsible for locking your apartment door, not leaving candles burning, and who you invite inside (**Security IN the Cloud**).

When you run an Amazon EC2 server:
- AWS ensures the physical server hardware and power don't fail.
- You ensure your software is updated, your firewall ports are secure, and your passwords/keys are protected!


In [ ]:
# Shared Responsibility Matrix Checker
def who_is_responsible(component: str) -> str:
    component_lower = component.lower()
    if any(k in component_lower for k in ["physical", "datacenter", "hardware", "power", "cooling", "nitro"]):
        return "AWS Responsibility (Security OF the Cloud)"
    elif any(k in component_lower for k in ["os patch", "code", "password", "firewall", "data encryption", "iam"]):
        return "Customer / Developer Responsibility (Security IN the Cloud)"
    else:
        return "Shared / Service Dependent"

print("• Physical Server Power :", who_is_responsible("Physical Server Power"))
print("• Linux OS Security Patch:", who_is_responsible("Linux OS Security Patch"))
print("• Application Code Bugs :", who_is_responsible("Application Code Bugs"))
print("• Datacenter Perimeter  :", who_is_responsible("Datacenter Perimeter Gates"))


---
### 🛡️ Step 4: The Zero-Spend $1.00 Budget Shield
Every beginner worries: *"What if I accidentally leave a server running and get a big bill?"*

To solve this on Day 1, we set up **The $1.00 Budget Shield**:
- Free Tier gives you **750 hours/month** of micro instances (runs 1 server 24/7 for free).
- We create an automated AWS Budget set to **$1.00 USD**.
- We configure an automated email alert at **80% ($0.80)**.
- If actual OR forecasted spend crosses 80 cents, AWS instantly emails you!


In [ ]:
def calculate_budget_shield(running_servers: int, days: int = 30):
    """Calculates monthly hours, charges, and shield alert triggers."""
    FREE_TIER_HOURS = 750.0
    HOURLY_RATE = 0.0116 # t3.micro in Mumbai
    BUDGET_LIMIT = 1.00
    ALERT_THRESHOLD = 0.80 # 80%
    
    total_hours = running_servers * 24 * days
    billable_hours = max(0.0, total_hours - FREE_TIER_HOURS)
    projected_bill = billable_hours * HOURLY_RATE
    alarm_fired = projected_bill >= ALERT_THRESHOLD
    
    return {
        "servers": running_servers,
        "total_hours": total_hours,
        "free_tier_allowed": FREE_TIER_HOURS,
        "billable_hours": billable_hours,
        "projected_bill_usd": f"${projected_bill:.2f}",
        "shield_alarm_triggered": alarm_fired
    }

print("Scenario A (1 Server 24/7):")
print(calculate_budget_shield(1))

print("\nScenario B (2 Servers 24/7 - Exceeds Free Tier):")
print(calculate_budget_shield(2))


---
### 📄 Step 5: Generating the Official AWS Budgets JSON Payload
Let's use Python's built-in `json` module to generate the exact JSON configuration payload you can feed to the AWS CLI or AWS Console:


In [ ]:
import json

def generate_budget_json(student_email: str):
    budget_payload = {
        "Budget": {
            "BudgetName": "CloudDeveloper-OneDollarShield",
            "BudgetLimit": {"Amount": "1.00", "Unit": "USD"},
            "TimeUnit": "MONTHLY",
            "BudgetType": "COST"
        },
        "NotificationsWithSubscribers": [
            {
                "Notification": {
                    "NotificationType": "ACTUAL",
                    "ComparisonOperator": "GREATER_THAN",
                    "Threshold": 80.0,
                    "ThresholdType": "PERCENTAGE"
                },
                "Subscribers": [
                    {"SubscriptionType": "EMAIL", "Address": student_email}
                ]
            }
        ]
    }
    return json.dumps(budget_payload, indent=2)

print("Generated AWS Budgets JSON (Ready for AWS CLI v2):")
print(generate_budget_json("student@college.edu"))


---
### 🏆 Day 01 Milestone Complete!
Look at what you have accomplished today:
1. **Python Foundations**: Syntax, variables as sticky labels, memory inspection with `id()`, strings, slicing, f-strings, lists, dictionaries, tuples, sets, decisions, loops, and functions.
2. **AWS Cloud Foundations**: The electric grid mental model, Regions, AZs, Mumbai & Hyderabad infrastructure, and the Shared Responsibility model.
3. **Financial Protection**: The $1.00 Budget Shield is verified and ready.

In **Day 02**, we will install the **AWS CLI v2** in our terminal, configure our credentials, and start running cloud commands directly from our command prompt! 🚀
